# Workflow ngắn gọn
1. **Baseline**: Lấy **trung bình theo gene** của toàn bộ training → vector trung tính μ.
2. **Align vocab**: Map gene dữ liệu ↔ vocab scGPT; gene thiếu dùng **mean embedding** (fallback).
3. **Cosine diffusion**:
    - Knockdown trực tiếp gene đích t: μ_t ← k * μ_t với `KNOCKDOWN_FACTOR=k`.
    - Tính **cosine similarity** giữa gene đích và **mọi** gene còn lại trong embedding scGPT; cắt âm về 0, chuẩn hoá thành trọng số w.
    - "Rò" một phần α của lượng giảm ở t sang hàng xóm: μ_i ← μ_i - α * (1-k) * old(μ_t) * w_i. (`ALPHA = α`)
4. **Sinh cell-level + khớp UMI**: Nhân nhiễu log-normal để có profile theo cell, rồi **rescale** về **median UMI** yêu cầu của từng perturbation.
5. **Gộp & xuất**: Nối tất cả perturbation + **5 000 control cells**, xuất **H5AD** theo đúng thứ tự/đặc tả.

In [ ]:
!python -m gdown --fuzzy --folder -O pretrained_scgpt https://drive.google.com/drive/folders/1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y

In [ ]:
ALPHA = 0.12            # how strongly we diffuse the target gene through scGPT embeddings
KNOCKDOWN_FACTOR = 0.55   # multiplicative knockdown for the perturbed gene
NOISE_SIGMA = 0.18        # log-normal sigma for per-cell sampling
CONTROL_CELLS = 5000      # extra non-targeting controls added to the submission
CHUNK_SIZE = 4000         # streaming chunk size when reading the training AnnData

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from scipy.sparse import issparse

DATA_DIR = Path("vcc_data")
MODEL_DIR = Path("pretrained_scgpt")

TRAINING_FILE = DATA_DIR / "adata_Training.h5ad"
VALIDATION_SPEC = DATA_DIR / "pert_counts_Validation.csv"

ALPHA = 0.12
KNOCKDOWN_FACTOR = 0.55
NOISE_SIGMA = 0.18
CONTROL_CELLS = 5000
CHUNK_SIZE = 4000


def load_scgpt_embeddings() -> Tuple[Dict[str, int], np.ndarray]:
    """Load vocab and embedding matrix from the local scGPT checkpoint."""
    with open(MODEL_DIR / "vocab.json", "r") as handle:
        vocab = json.load(handle)
    state_dict = torch.load(MODEL_DIR / "best_model.pt", map_location="cpu")
    embedding = state_dict["encoder.embedding.weight"].cpu().numpy().astype(np.float32)
    return vocab, embedding



def align_embeddings(gene_names: Iterable[str],
                     vocab: Dict[str, int],
                     embedding_matrix: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Map challenge genes into the scGPT embedding space with a mean fallback."""
    gene_names = list(gene_names)
    default_embedding = embedding_matrix.mean(axis=0)
    aligned = np.zeros((len(gene_names), embedding_matrix.shape[1]), dtype=np.float32)
    indices = np.full(len(gene_names), -1, dtype=np.int32)
    missing = []

    for idx, gene in enumerate(gene_names):
        vocab_idx = vocab.get(gene)
        if vocab_idx is None:
            aligned[idx] = default_embedding
            missing.append(gene)
        else:
            indices[idx] = vocab_idx
            aligned[idx] = embedding_matrix[vocab_idx]

    return indices, aligned, np.array(missing)



def stream_mean_profile(adata_path: Path, chunk_size: int = CHUNK_SIZE) -> Tuple[np.ndarray, int]:
    """Accumulate the cell-mean baseline in streamed chunks to avoid dense loads."""
    adata = sc.read_h5ad(adata_path, backed="r")
    n_cells, n_genes = adata.shape
    running_sum = np.zeros(n_genes, dtype=np.float64)

    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        chunk = adata[start:end].X
        if issparse(chunk):
            chunk_sum = np.asarray(chunk.sum(axis=0)).ravel()
        else:
            chunk_sum = np.asarray(np.sum(chunk, axis=0)).ravel()
        running_sum += chunk_sum

    return (running_sum / max(n_cells, 1)).astype(np.float32), n_cells



def apply_embedding_influence(base_profile: np.ndarray,
                              target_idx: int,
                              aligned_embeddings: np.ndarray,
                              embedding_norms: np.ndarray) -> np.ndarray:
    """Diffuse the perturbation through cosine similarity in embedding space."""
    target_emb = aligned_embeddings[target_idx]
    target_norm = embedding_norms[target_idx]
    influence = (aligned_embeddings @ target_emb) / (embedding_norms * target_norm + 1e-6)
    influence = np.clip(influence, -1.0, 1.0).astype(np.float32)
    adjusted = base_profile * (1.0 + ALPHA * influence)
    adjusted[target_idx] *= KNOCKDOWN_FACTOR
    return np.clip(adjusted, 0.0, None)



def sample_cells(base_profile: np.ndarray,
                 n_cells: int,
                 median_umi: float | None,
                 rng: np.random.Generator) -> np.ndarray:
    """Sample log-normal noise around the perturbed mean and enforce UMI totals."""
    if n_cells <= 0:
        return np.zeros((0, base_profile.size), dtype=np.float32)

    tiled = np.tile(base_profile, (n_cells, 1))
    noise = rng.lognormal(mean=0.0, sigma=NOISE_SIGMA, size=tiled.shape).astype(np.float32)
    sampled = np.clip(tiled * noise, 0.0, None)

    if median_umi and median_umi > 0:
        totals = sampled.sum(axis=1)
        valid = totals > 0
        if valid.any():
            sampled *= median_umi / np.median(totals[valid])

    return sampled.astype(np.float32)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad    
import json

OUTPUT_DIR = Path("scgpt_runs")
OUTPUT_NAME = "submission_scgpt_baseline.h5ad"

if True:  # flip to True to regenerate the full submission (takes ~3 minutes)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    validation = pd.read_csv(VALIDATION_SPEC)
    training_mean, _ = stream_mean_profile(TRAINING_FILE)

    adata_ref = sc.read_h5ad(TRAINING_FILE, backed="r")
    gene_names = adata_ref.var_names.tolist()

    vocab, embedding_matrix = load_scgpt_embeddings()
    _, aligned_embeddings, missing_genes = align_embeddings(gene_names, vocab, embedding_matrix)
    embedding_norms = np.linalg.norm(aligned_embeddings, axis=1).astype(np.float32)
    embedding_norms[embedding_norms == 0.0] = 1.0

    gene_name_to_idx = {gene: idx for idx, gene in enumerate(gene_names)}
    rng = np.random.default_rng(seed=1234)
    predictions: Dict[str, np.ndarray] = {}

    for _, row in validation.iterrows():
        gene = row["target_gene"]
        n_cells = int(row["n_cells"])
        median_umi = float(row["median_umi_per_cell"])

        profile = training_mean.copy()
        idx = gene_name_to_idx.get(gene)
        if idx is not None:
            profile = apply_embedding_influence(profile, idx, aligned_embeddings, embedding_norms)
        else:
            profile *= 0.98  # soft fallback for missing genes

        predictions[gene] = sample_cells(profile, n_cells, median_umi, rng)

    if "non-targeting" not in predictions:
        control_profile = training_mean.copy()
        control_umi = float(validation["median_umi_per_cell"].median())
        predictions["non-targeting"] = sample_cells(control_profile, CONTROL_CELLS, control_umi, rng)

    matrices = []
    obs_records = []
    for gene, matrix in predictions.items():
        matrices.append(matrix.astype(np.float16))
        obs_records.extend({"target_gene": gene, "cell_id": f"{gene}_{i}"} for i in range(matrix.shape[0]))

    X = np.vstack(matrices)
    obs = pd.DataFrame(obs_records)
    adata_out = ad.AnnData(X=X, obs=obs)
    adata_out.var_names = gene_names
    adata_out.obs_names = obs["cell_id"].astype(str)

    output_path = OUTPUT_DIR / OUTPUT_NAME
    adata_out.write_h5ad(output_path, compression="gzip")
    print(f"Saved to {output_path}")
else:
    print("Set the guard to True to regenerate the submission; default is documentation-only.")

In [ ]:
!python -m cell_eval prep -g "vcc_data/gene_names.csv" -i "scgpt_runs/submission_scgpt_baseline.h5ad" -o "submissions/scgpt_02.vcc"